<font color=red>**Danger zone:**</font> you'll be fine-tuning a model to generate positive, negative or even toxic reviews. We'll be doing this for fun, but this is also the technique for [review bombing](https://en.wikipedia.org/wiki/Review_bomb), bot farms on social media and other less than dignified stuff. It is ultimately your decision how you apply this knowledge, but before you choose, ask yourself: is this why you chose to learn ML?


# LLMs Alignment with Reinforcement Learning from human feedback (RLHF).

_based on the [original notebook](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) by Ilya Boytsov for the Oxford LLMs workshop_



In this session, you're gonna fine-tune a language model with reinforcement learning to make it generate good (or bad) reviews.

To perform RL-based fine-tuning, we'll use a new (in this course) library called [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). TRL implements the main reinforcement learning components of RLHF: reward modeling and fine-tuning with PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

In [ ]:
%pip install -q trl==0.11.0 transformers==4.45.2 datasets==3.4.1 peft==0.14.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


### Tutorial: align the model to generate positive movie reviews

To see how TRL works, we'll use it to align GPT2 on IMDB dataset to generate positive (or negative) movie reviews. In fact, __it's your choice whether you want positive or negative reviews.__

But before you choose, let's take a look at the baseline model: a GPT-2 fine-tuned on generating arbitrary movie reviews.

In [ ]:
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



Generated text: The movie could have saved its time by being less predictable, but not.<br /><br />In my view, most directors would have liked to use a somewhat more sophisticated approach to the plot. One thing that I liked about the movie was that most of


If you run this cell a couple of times, you'll see that the model generates both positive, negative and neutral reviews in some proportion. What we're gonna do next is teach the model to generate more positive (or negative) reviews.

Similarly to InstructGPT, we're gonna do that in 2 stages:
- **train a reward model** to assign higher values to positive (or negative) reviews
- fine-tune the language model to **maximize that reward using [proximal policy optimization](https://openai.com/research/openai-baselines-ppo)**



## Stage 1: train a reward model

First, we'll train a BERT-like model as our reward model. We'll generate a synthetic pairwise rankings to emulate human rankings.

__Q:__ why do I need a reward model? Can I just use a pre-trained sentiment classifier? <br> __A:__ Yes, you can - but that only works for movie reviews. But this tutorial will teach you how to do RLHF for any kind objective.


__If you actually want to maximize sentiment (or other "label") instead of human preferences, train reward model as a classifier! (see week5)__


In [ ]:
# We'll be fine-tuning a small BERT-like model for now. Please try other models for the main assignment.
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

__Note that__ the reward model has a separate tokenizer, different from the main model. They don't need to be the same for RLHF fine-tuning.

In [ ]:
# To train a reward model, you need a dataset (or generator) of positive-negative pairs.
# Each training sample should be a dict with 4 keys:
#  - input_ids_chosen, attention_mask_chosen = tokenizer("A sentence that human labeler likes more")
#  - input_ids_rejected, attention_mask_rejected = tokenizer("A sentence that human labeler likes less")

import torch
import datasets

class IMDBPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, imdb, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['text'] for row in imdb if row['label'] == accepted_label]
        self.rejected_texts = [row['text'] for row in imdb if row['label'] != accepted_label]
        self.column_names = ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected']
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [ ]:
TARGET_LABEL = 0  # and make sure it works by reviewing the sample printed below
imdb = datasets.load_dataset("imdb", split='train')
reward_data = IMDBPairwiseDataset(imdb, reward_tokenizer, accepted_label=TARGET_LABEL)

sample = reward_data[31337]
print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Found 12500 chosen and 12500 rejected texts, 156250000 pairs
CHOSEN: [CLS] If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story. < br / > < br / > One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives ( unless one comes up with one while one ' s mind wanders, as it will invariably do during this pointless film ). < br / > < br / > One might better spend one ' s time staring out a window at a tree growing. < br / > < br / > [SEP]
REJECTED: [CLS] This movie has some things that are pretty amazing. First, it is supposed to be based on a true story. That, in itself, is amazing that multiple tornadoes would hit the same town at night in the fall - in Nebraska. I wonder if the real town ' s name was close to " Blainsworth " ( which is the town ' s name in the movie ). There is an Ainsworth, N

We'll be using `trl.RewardTrainer` - a special case of `transformers.Trainer` that you used in the past. `RewardTrainer` accepts the same format of training arguments (e.g. batch size, gradient checkpointing) as before, except that it trains the model for the pairwise reward objective from [the InstructGPT paper](https://arxiv.org/pdf/2203.02155.pdf):

![img](https://i.imgur.com/2JzNAPs.png)

Note that the model itself does not score pairs: it processes chosen ($y_w$) and rejected ($y_l$) samples independently. To minimize this loss, the reward model needs to score chosen sample higher than the rejected one. Note that the formula also assumes some context $x$, which is useful for seq2seq tasks. In our case of movie reviews, $x$ is empty.

In [ ]:
import trl
import os


training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,                     # disable this on CPU or on very old GPUs
    # you may add any other hyperparameters that you found useful in weeks 5-7
    report_to="none",
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    tokenizer=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc
)


trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`T

Step,Training Loss
50,0.551300
100,0.192500
150,0.135800
200,0.118100
250,0.091900
300,0.099800
350,0.095600
400,0.093600
450,0.082100
500,0.072900


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=0.10918231415748596, metrics={'train_runtime': 1525.245, 'train_samples_per_second': 20.98, 'train_steps_per_second': 0.656, 'total_flos': 0.0, 'train_loss': 0.10918231415748596, 'epoch': 0.00020479997902848215})

In [ ]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

### Sanity-check the reward model (1 point)

Let's check how our reward model performs.

__Your task__ is to measure how often does your reward model can rank a pair of (chosen and rejected) reviews correctly. Please measure this separately for train data (`imdb`) and a separate test set loaded below.

In [ ]:

for sample_index in 45, 16000:
  print('TEXT:', imdb[sample_index]['text'])
  inputs = reward_tokenizer(
      imdb[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', imdb[sample_index]['label'])
  print()

# note: your reward model may produce different absolute rewards.
# This is fine as long as the rewards are ordered correctly (most of the time)

TEXT: This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get "terrorized" by this pathetic "crazed killer", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.
REWARD: 5.00390625
LABEL: 0

TEXT: Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes on.<br /

In [ ]:
imdb_test = datasets.load_dataset("imdb", split='test')

# <a whole lot of your code here, feel free to spit it as you see fit>
reward_data_test = IMDBPairwiseDataset(imdb_test, reward_tokenizer, accepted_label=TARGET_LABEL)

Found 12500 chosen and 12500 rejected texts, 156250000 pairs


In [ ]:
from tqdm.auto import tqdm


reward_model.eval()
with torch.no_grad():
    correct = 0
    for i, row in tqdm(enumerate(reward_data_test)):
        chosen_input_ids = torch.tensor(row["input_ids_chosen"]).unsqueeze(0).to(device)
        chosen_att_mask = torch.tensor(row["attention_mask_chosen"]).unsqueeze(0).to(device)
        chosen_out = reward_model(
            input_ids=chosen_input_ids,
            attention_mask=chosen_att_mask,
        )

        rejected_input_ids = torch.tensor(row["input_ids_rejected"]).unsqueeze(0).to(device)
        rejected_att_mask = torch.tensor(row["attention_mask_rejected"]).unsqueeze(0).to(device)
        rejected_out = reward_model(
            input_ids=rejected_input_ids,
            attention_mask=rejected_att_mask,
        )

        if chosen_out.logits[0,0].item() > rejected_out.logits[0,0].item():
            correct += 1

        if i >= 16000:
            break
# should be 16000 not len(reward_data_set)
print(f"Ranked {correct} out of {len(reward_data_test)} pairs correctly.")

0it [00:00, ?it/s]

Ranked 15986 out of 156250000 pairs correctly.


In [ ]:
print(f"Ranked {correct} out of 16000 pairs correctly.")

Ranked 15986 out of 16000 pairs correctly.


### Reward-guided generation (1 point)

If you did everything right, by now you should have a decent reward model. Before we use it for reinforcement learning, let's see if we can align model samples without any training.

To do so, you can use reward-guided inference: __generate N=16 samples, then select the one with the highest reward__ (according to your reward model).

For this problem, it's on you to demonstrate whether or not your code works. Find at least 5 neutral prompts such as "This movie is" (...), generate samples, rank them based on reward and show which samples get the highest reward.

Note: it is faster to generate samples in parallel, rather than sequentially, as follows:




In [ ]:
inputs = main_tokenizer(["It was"] * 5, return_tensors='pt').to(device)
for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
  print("Sample:", main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was also filmed on stage with actors from other movies, although some, like "Babylon 5" and "The Patriot", feature characters of different nationalities. All told, it's an interesting and well-written film, and one of films where
Sample: It was hard to believe that the film would be made on DVD by a company that was trying to build up "Star Trek IV: The Voyage Home", which became a successful franchise and is quite successful after so many great movies.<br /><br />
Sample: It was like the second time you watch TV in the world. But you are bored and want a movie.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was a good movie. N

In [ ]:
# <YOUR CODE HERE> - feel free to organize it as you see fit
# <YOUR CODE HERE> - feel free to organize it as you see fit
def sample_positive(prompt, n_samples=16, print_all_samples=True):
    """Generate n_samples samples and choose the best ranked one by reward_model"""
    inputs = main_tokenizer([prompt] * n_samples, return_tensors='pt').to(device)
    candidates = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)

    samples=[]
    for candidate in candidates:
        sample = main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist())

        samples.append(sample)

        if print_all_samples:
            print(f"Sample: {sample}")

    out = reward_model(**reward_tokenizer(samples, padding="max_length", truncation=True, max_length=512, return_tensors="pt").to(device))
    best_id = torch.argmax(out.logits[:, 0])

    print(f"BEST: {samples[best_id]}")

prompts = ["This movie is", "It was", "I thought", "From the beginning", "This experience"]

for prompt in prompts:
    sample_positive(prompt)
    print()


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: This movie is full of humor and I would have gladly left that film. It's fun stuff. However, it lacks most of the charm and the story is a mess. There are a few good jokes that get thrown out of the script and then forgotten, but
Sample: This movie is very good and will not make the world any more predictable. We see a young man (who, unlike the bad guys, does not have a clue that he's being treated here) coming to the aid of a child who seems to be in need
Sample: This movie is full of great action, great dialogue and a great storyline by Michael Moore. The movie is very funny and I am sure this movie is one of the best parts of the movie.<br /><br />Now here is the thing I must say about
Sample: This movie is great, but I would rather have someone on the phone to explain that it was the stupidest thing of the whole movie...I just can't believe someone would do such a stupid thing to make a bad movie. For those of you who know anything
Sample: This movie is so boring and unori

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was a pretty good picture of this kind of movie.<br /><br />Now, I hate to say this, but it's a pretty good movie. The plot itself is good, but you have to admit the casting of the main character to hold
Sample: It was the movie's first real theatrical release. In fact, in the end, Paramount Pictures gave it a theatrical release: I can't imagine why?<br /><br />The title alone could be construed as a warning that a movie should be made
Sample: It was a good movie. When I saw this movie it's so sad... It's sad because I'm not sure how I want to watch it again.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was in the 70s, even among the most educated white kids who would not mind a bad-ass chick. That means you're probably as good off today as you were bac

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: I thought it was really funny. It was just that I was hooked.<br /><br />I hope people aren't having too much of a "what the hell you talking about! How were you on the set?", because if you did and didn't
Sample: I thought "that's a good idea!<br /><br />I would have preferred something more sensible, like an explicit sex scene between two girls. I think that the way the plot is presented is brilliant.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: I thought I'd read somewhere, but it was a waste of time. It's just plain bad. It makes you think: How is it that a lot of people like this? I actually liked it, and the plot never gets boring, but I did
Sample: I thought I would love to see it back. So...it was...I got it!<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|en

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: From the beginning, the movie is about a woman trying to be a starlet, but she discovers that she has a little body. Then when a movie starts with a girl with a lot of power, all the girls around her face go mad, causing the movie
Sample: From the beginning of the film there is a bit of a cliche about one of the "jokes", but ultimately this takes place in a somewhat different setting where the characters don't quite have it that way. It also sets up very interesting relationships between characters as
Sample: From the beginning, it was pretty easy to watch; a typical teenish-movie-like scenario of a bad family, drugs, and alcohol. As far as "family dynamics", I liked where we started talking about how the kids were dysfunctional - how they acted
Sample: From the beginning you expect something really strange inside of you to happen to get an answer, but it doesn't. Like I said before, it's just a bunch of people trying to make a little bit of money off of the film, and then at t

# Stage 2: fine-tune the main model with RL


For this tutorial, we will optimize GPT2 to produce positive IMDB movie reviews using the reward model you trained above.

Unlike supervised fine-tuning, RL allows model to generate it's own sentences on each training step. Then, it calculates the reward of those specific sentences, and finally, updates the model to increase the probability of sentences with high reward.

Thus, each RLHF consists of three stages: __Rollout__, __Evaluation__ and __Update__

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

The update stage depends on the specific RL algorithm. We'll be using Proximal Policy Optimization, or [PPO](https://arxiv.org/abs/1707.06347), similarly to what was used for InstructGPT.

Before we run those 3 stages, however, we need to create a dataset of "queries" - partial reviews in our case.

In [ ]:
from trl.core import LengthSampler

In [ ]:
# Note: this code is specific to IMDB; you will need to re-write it for other tasks
imdb_for_rlhf = imdb.filter(lambda row: len(row['text']) > 200, batched=False)
imdb_for_rlhf = imdb_for_rlhf.remove_columns(['label'])
sample_length = LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

imdb_for_rlhf = imdb_for_rlhf.map(select_query_and_tokenize, batched=False)
imdb_for_rlhf.set_format(type="torch")

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/24895 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


Next, let's prepare your reward model to predict rewards on whatever reviews were generated. Note that we use plaintext reviews because main model uses a different tokenizer from the reward model.

In [ ]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [ ]:
compute_reward([imdb[45]['text'], imdb[16000]['text']])  # test on human-written reviews

tensor([ 5.0039, -4.8008], device='cuda:0')

Finally, we move to RL training. In this tutorial, we'll train LoRA adapters and not the full model.

In [ ]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9391


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:1264: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Same as before, trl has a special type of trainer that minimize PPO-specific pseudo-loss. You can read more on this trainer [here](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [ ]:
training_args = trl.PPOConfig(
    model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    batch_size=64,
    mini_batch_size=4,
    ppo_epochs=4,                 # PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=main_model.model, tokenizer=main_tokenizer,
    dataset=imdb_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(


In [ ]:
from tqdm.auto import tqdm
max_steps = 50   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=128, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Rollout stage: generate continuations from batch queries using main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # de-tokenize responses to strings (since reward model uses a different tokenizer)
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # note: response_tensors already contain query tokens, so we don't need to add queries manually.
    # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]


    # Evaluation stage
    rewards = compute_reward(batch['response'])

    # Update stage
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  0%|          | 0/50 [00:00<?, ?it/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


------------------------------ STEP 0 ------------------------------
rewards/mean:	-0.060281754	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.619707465	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)

------------------------------ STEP 1 ------------------------------
rewards/mean:	0.710150003	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.439525962	<---- model-estimated average discounted reward
objective/kl:	-0.090662643	<---- how far we are from the original model (regularizer)

------------------------------ STEP 2 ------------------------------
rewards/mean:	-0.029399872	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.474836648	<---- model-estimated average discounted reward
objective/kl:	-0.031037735	<---- how far we are from the original model (regularizer)

------------------------------ STEP 

## Main assignment - <u>actually</u> train the model (8 points)


Your main task for this week is to use the RLHF pipeline to train a model for a reward of your choice. Here's what you can choose from:

__A. Toxicity fine-tuning:__ train the model to be less (or more!) toxic. For this task, you may use the data from [jigsaw toxic comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) and [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat),  or any other source. Alternatively, you may use toxicity scores from [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).


#### General tips & tricks


Things to look out for:
- during PPO stage, the reward model should be in eval mode (dropout disabled)
- make sure max_length and max_new_tokens are enough for your chosen dataset - at least most of the time
- when in doubt, view the data manually or inspect how the model performs on a few samples


We highly recommend that you manually check the performance after each sub-stage:
1. when you assembled the pairwise dataset, inspect a couple of from of *your* dataset class and detokenize them. Make sure that you-the-human understand why one sample was accepted and the other - rejected. At least most of the time. This also lets you spot tokenization/truncation errors.
2. after you trained a reward model, measure how accurate this model is in isolation. If your reward model is poor, any subsequent RLHF will also fail.
3. once you've trained the main model with RL, ask it to generate examples and explore how well it does. If it produces an obviously bad output, check if the reward model assigns high reward to that output. If yes, reward model is the culprit; if no, it's a question of better/longer PPO training.

__It is also a good idea to periodically print samples during training.__

__When stuck, simplify the problem.__ If you've spent a several hours enchanting the reward model but it still won't budge, try switching to a simple subtask. For instance, if you're training on hh-rlhf, try limiting it the dataset to 10% of the shortest sequences - they are typically easier to learn.


## Assignment stages (and grading)

Regardless of the specific task you chose, your solution needs to contain several parts that will be graded separately.


#### Stage 1: reward model (4 points)

Construct a dataset for training the reward model on your problem. Then, train a reward model on that dataset and evaluate how well can your model predict preferences on a hold-out (test) subset of your data.

Please make sure that the part of your notebook where you evaluate reward model is clearly visible and reasonably easy to read. And for all that is holy, do not call it IMDB unless it actually **is** data of imdb movie reviews :)

__Not all tasks require a reward model for later PPO fine-tuning.__ For instance, there's no reason to train a reward model if your reward equals sentence length. Likewise, toxicity reward can be estimated with a pre-trained toxicity classifier. __If your task does not require training a reward model, please train an unrelated model on [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) as though you were solving assignment version B.__ This is for grading purposes only, you won't use this model for stage 2.


#### Stage 2: RL fine-tuning (4 points)

Once the reward model is ready - or you can compute rewards without a model - it is time to maximize that reward with PPO. Optionally, you may replace PPO with another RL algorithm (or unlikelihood learning scheme), but only if you're feeling adventurous.


First, you need to choose a language model to be fine-tuned. You may choose any model, but make sure that your model **can** generate the data in your format. For instance, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) is a general purpose LM and may (or may not) need prompt engineering to generate chat assistant responses. For that reason, it is best if you **do not use `"lvwerra/gpt2-imdb"` unless you're generating only movie reviews**.



There are two "difficulty modes" for this task:
For the **easy mode**, use [gpt2-large](https://huggingface.co/gpt2-large) or [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) with minimal code changes.
If you want the **Hard mode:** use a larger (e.g. 7B) model in combination with `load_in_4bit` and LoRA, the same way we did last week.
Some reasonable model choices are [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7b](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) for general-purpose LM or [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b), [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) for chat-based tasks, though there are many more (see [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)). In the hard mode, you will need to modify the training arguments to enable 4-bit fine-tuning. Furthermore, your experiments will take somewhat longer to complete. On the plus side, your model will produce significantly better results.

__High reward is not enough!__ RL algorithms are famous for [cheating their reward functions](https://openai.com/research/faulty-reward-functions). To ensure that your model is actually doing what you want it to do, you will need some additional evaluation. To get the full grade, provide at least 20 side-by-side examples of your fine-tuned model vs original model predictions and a short summary.

Alternatively, you may provide 5 examples and some extrinsic evaluation metric over many examples. For instance, you may use a different pre-trained toxicity score for option A. When dealing with human preferences, you may choose to [enlist actual humans](https://toloka.ai/) or [ask GPT4/Claude](https://arxiv.org/pdf/2304.03277.pdf) to compare your model's predictions. For task C, when optimizing for simple rewards like sentence lengths, it is enough to compare histograms of rewards (e.g. average lengths).












In [ ]:
%pip install -q trl==0.11.0 transformers==4.45.2 datasets==3.4.1 peft==0.14.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


In [ ]:
import torch
import transformers
import pandas as pd
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
main_tokenizer = transformers.AutoTokenizer.from_pretrained("openai-community/gpt2-large")
main_model = transformers.AutoModelForCausalLM.from_pretrained("openai-community/gpt2-large", device_map=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
inputs = main_tokenizer("I don't like", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



Generated text: I don't like going home, except with my child, to say goodbye, but in terms of a personal relationship, it is very real, even if it's not in love."

But the fact that his wife is in charge of the household makes it difficult


## Training reward model

In [ ]:
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
import datasets

comments = datasets.load_dataset("tasksource/jigsaw_toxicity")
comments = comments["train"]

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

In [ ]:
comments

Dataset({
    features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
    num_rows: 159571
})

In [ ]:
label_column = [0] * len(comments)
comments = comments.add_column("label", label_column)

features = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

def set_label(row):
    label = 0
    for feat in features:
        if row[feat]:
            label = 1
            break

    row["label"] = label
    return row

comments = comments.map(set_label, remove_columns=features, num_proc=4)

Map (num_proc=4):   0%|          | 0/159571 [00:00<?, ? examples/s]

In [ ]:
comments_pd = comments.to_pandas()

label0 = comments_pd[comments_pd["label"] == 0][:16225] # 16225 is the number of comments with label == 0
label1 = comments_pd[comments_pd["label"] == 1][:16225]

train_len = 12000

train_pd = pd.concat([label0[:12000], label1[:12000]])
test_pd = pd.concat([label0[12000:], label1[12000:]])

comments_train = datasets.Dataset.from_pandas(train_pd, preserve_index=False)
comments_test = datasets.Dataset.from_pandas(test_pd, preserve_index=False)

In [ ]:
comments_train, comments_test

(Dataset({
     features: ['id', 'comment_text', 'label'],
     num_rows: 24000
 }),
 Dataset({
     features: ['id', 'comment_text', 'label'],
     num_rows: 8450
 }))

In [ ]:
class JigsawPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, data, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['comment_text'] for row in data if row['label'] == accepted_label]
        self.rejected_texts = [row['comment_text'] for row in data if row['label'] != accepted_label]
        self.column_names = ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected']
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [ ]:
TARGET_LABEL = 1
reward_data = JigsawPairwiseDataset(comments_train, reward_tokenizer, accepted_label=TARGET_LABEL)

sample = reward_data[31337]
print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

Found 12000 chosen and 12000 rejected texts, 144000000 pairs
CHOSEN: [CLS] Bye! Don ' t look, come or think of comming back! Tosser. [SEP]
REJECTED: [CLS] laughingstock PC redirection COVER THE EVENT NOT THE PERSON. Whipple was a women ' s lacrosse player, not a notable athlete by any stretch. She tried to make the olympics and failed. In fact, very few people would have heard of this woman if she hadn ' t got into a tiff with Marjorie Knoeller that fateful day. This article leaves out many of the important details and is laughable. Marjorie Knoeller should not redirect here. [SEP]


In [ ]:
import trl
import os


training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=500,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,                     # disable this on CPU or on very old GPUs
    # you may add any other hyperparameters that you found useful in weeks 5-7
    report_to="none"
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    tokenizer=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc
)


trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`T

Step,Training Loss
50,0.528500
100,0.193200
150,0.142400
200,0.120300
250,0.091300
300,0.095200
350,0.107200
400,0.094300
450,0.060800
500,0.093800


TrainOutput(global_step=500, training_loss=0.1526951336860657, metrics={'train_runtime': 146.8806, 'train_samples_per_second': 27.233, 'train_steps_per_second': 3.404, 'total_flos': 0.0, 'train_loss': 0.1526951336860657, 'epoch': 2.777777777777778e-05})

In [ ]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
for sample_index in 6, 16000:
  print('TEXT:', comments[sample_index]['comment_text'])
  inputs = reward_tokenizer(
      comments[sample_index]['comment_text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', comments[sample_index]['label'])
  print()

TEXT: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK
REWARD: 3.55078125
LABEL: 1

TEXT: Vankenta here: Well, it seems that there is no hope for .
REWARD: -3.693359375
LABEL: 0



In [ ]:
reward_data_test = JigsawPairwiseDataset(comments_test, reward_tokenizer, accepted_label=TARGET_LABEL)

Found 4225 chosen and 4225 rejected texts, 17850625 pairs


In [ ]:
from tqdm.auto import tqdm

NUM_PAIRS = 16000 # number of pairs to test

reward_model.eval()
with torch.no_grad():
    correct = 0
    for i, row in tqdm(enumerate(reward_data_test)):
        chosen_input_ids = torch.tensor(row["input_ids_chosen"]).unsqueeze(0).to(device)
        chosen_att_mask = torch.tensor(row["attention_mask_chosen"]).unsqueeze(0).to(device)
        chosen_out = reward_model(
            input_ids=chosen_input_ids,
            attention_mask=chosen_att_mask,
        )

        rejected_input_ids = torch.tensor(row["input_ids_rejected"]).unsqueeze(0).to(device)
        rejected_att_mask = torch.tensor(row["attention_mask_rejected"]).unsqueeze(0).to(device)
        rejected_out = reward_model(
            input_ids=rejected_input_ids,
            attention_mask=rejected_att_mask,
        )

        if chosen_out.logits[0,0].item() > rejected_out.logits[0,0].item():
            correct += 1

        if i >= NUM_PAIRS:
            break
# should be 16000 not len(reward_data_set)
print(f"Ranked {correct} out of {NUM_PAIRS} pairs correctly. Acc {correct / NUM_PAIRS}")

0it [00:00, ?it/s]

Ranked 15783 out of 16000 pairs correctly. Acc 0.9864375


In [ ]:
# <YOUR CODE HERE> - feel free to organize it as you see fit
# <YOUR CODE HERE> - feel free to organize it as you see fit
def sample_positive(prompt, n_samples=16, print_all_samples=True):
    """Generate n_samples samples and choose the best ranked one by reward_model"""
    inputs = main_tokenizer([prompt] * n_samples, return_tensors='pt').to(device)
    candidates = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)

    samples=[]
    print("="*50)
    for candidate in candidates:
        sample = main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist())

        samples.append(sample)

        if print_all_samples:
            print(f"Sample: {sample}")

    out = reward_model(**reward_tokenizer(samples, padding="max_length", truncation=True, max_length=512, return_tensors="pt").to(device))
    best_id = torch.argmax(out.logits[:, 0])

    print(f"BEST: {samples[best_id]}")
    print("="*50)

prompts = ["She was a", "I hope you", "It was the last", "From the beginning", "This experience"]

for prompt in prompts:
    sample_positive(prompt)
    print()


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: She was a brilliant actress,' wrote his sister to his brother Robert during the first days of September 1917. 'Always, as I read to her [he] never stopped smiling.'

On Monday, Ms Taylor had gone about the typical school routine of heading to
Sample: She was a model for the young lady who is now in her early 40s or early 50s—she's a wonderful model, and she has been for a long time, and she made a career of it—and she was a model when I was in
Sample: She was a great girl, she was always there for us, she was just always there."

The family was devastated and still struggling to come to grips with their loss.

"It's very, very sad, it's very hard," said Ms
Sample: She was a little bit nervous about what it would be like. But I've actually never been really nervous like that about anything, because I tend to be very patient and gentle with them when I get together, and they are very patient with me, and they understand
Sample: She was a good person to me," said Scott, who was on

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: I hope you enjoy the game so far. If you have any feedback, please leave any suggestions, comments, etc. I will be sure to continue developing it.


I'm really hoping that the community will like the way the game looks, because I know
Sample: I hope you don't like me." The priest said, "I am your mother." Aghazada: "I don't need any woman... Mother and me just can't stop each other! Nothing is going to give me the feeling of her touch...
Sample: I hope you're doing well!

It's been a while, so I had to make some changes. It's been a while, so I had to make some changes. I love the community, and the feedback you guys give me. I always enjoy
Sample: I hope you can share some of that joy with the community. The Community is always looking for ways to support their community and we at Votemaster are happy to take part in it."

You can go to votemaster.com/l
Sample: I hope you will follow our progress," says one of the many banners posted at the entrance to the building.

But this 

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was the last thing she ever wanted to see.

"I don't remember what happened to my husband. Did he run into a house or an abandoned building?" asked the widow.

She could barely make it down the stairs at 8:30 a
Sample: It was the last game in a month, the final game before I left on vacation.

I did some more light boxing.

I also didn't fight the next game. Then when I got back to the hotel again, I noticed a red shirt was
Sample: It was the last time I wanted to think about such things."<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: From the beginning, the city was a dream. One that was seen as untouchable by the authorities. What started off as a dream now feels like an open wound.

"The city of Nuevo Laredo has been a symbol of hope
Sample: From the beginning, the state's anti-abortion policy was far more conservative than that of Planned Parenthood, which has been praised for providing better abortions and for providing more comprehensive services.

The state's anti-abortion policy has remained a central political issue in Iowa
Sample: From the beginning of 2012, we have been working to improve the service. One of the big issues was ensuring the new technology could handle more requests. For example, the website went through various updates to handle user activity in a way that ensured the user received the most
Sample: From the beginning, we knew that the only way to meet all of our ambitions was to offer them to the world. We didn't have any other options."

The company's current vision

A spokesperso

In [ ]:
from trl.core import LengthSampler


# Note: this code is specific to IMDB; you will need to re-write it for other tasks
comments_for_rlhf = comments.filter(lambda row: len(row['comment_text']) > 200, batched=False)
comments_for_rlhf = comments_for_rlhf.remove_columns(['label'])
sample_length = LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["comment_text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

comments_for_rlhf = comments_for_rlhf.map(select_query_and_tokenize, batched=False)
comments_for_rlhf.set_format(type="torch")

Filter:   0%|          | 0/159571 [00:00<?, ? examples/s]

Map:   0%|          | 0/81066 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2132 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [ ]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

tokenizer_config.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9391


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:1264: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
training_args = trl.PPOConfig(
    model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    batch_size=64,
    mini_batch_size=4,
    ppo_epochs=4,                 # PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=main_model.model, tokenizer=main_tokenizer,
    dataset=comments_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(


In [ ]:
from tqdm.auto import tqdm
max_steps = 75   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=128, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Rollout stage: generate continuations from batch queries using main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # de-tokenize responses to strings (since reward model uses a different tokenizer)
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # note: response_tensors already contain query tokens, so we don't need to add queries manually.
    # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]


    # Evaluation stage
    rewards = compute_reward(batch['response'])

    # Update stage
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  0%|          | 0/75 [00:00<?, ?it/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


------------------------------ STEP 0 ------------------------------
rewards/mean:	-1.783752441	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.924598038	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)

------------------------------ STEP 1 ------------------------------
rewards/mean:	-1.286801338	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.918289661	<---- model-estimated average discounted reward
objective/kl:	-0.007081750	<---- how far we are from the original model (regularizer)

------------------------------ STEP 2 ------------------------------
rewards/mean:	-1.678587914	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.985166669	<---- model-estimated average discounted reward
objective/kl:	-0.052448671	<---- how far we are from the original model (regularizer)

------------------------------ STEP

In [ ]:
model = ppo_trainer.model.pretrained_model

In [ ]:
# <YOUR CODE HERE> - feel free to organize it as you see fit
# <YOUR CODE HERE> - feel free to organize it as you see fit
def sample_positive(prompt, n_samples=16, print_all_samples=True):
    """Generate n_samples samples and choose the best ranked one by reward_model"""
    inputs = main_tokenizer([prompt] * n_samples, return_tensors='pt').to(device)

    with torch.no_grad():
        candidates = model.generate(**inputs, max_new_tokens=50, do_sample=True)

    samples=[]
    print("="*50)
    for candidate in candidates:
        sample = main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist(), skip_special_tokens=True)

        samples.append(sample)

        if print_all_samples:
            print(f"Sample: {sample}")

    with torch.no_grad():
        out = reward_model(**reward_tokenizer(samples, padding="max_length", truncation=True, max_length=512, return_tensors="pt").to(device))
    best_id = torch.argmax(out.logits[:, 0])

    print(f"BEST: {samples[best_id]}")
    print("="*50)

prompts = ["She was a", "I hope you", "It was the last", "From the beginning", "This experience"]

for prompt in prompts:
    sample_positive(prompt)
    print()


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: She was a lovely actress with beautiful beauty, but she was not perfect. Not enough to make me laugh.
Sample: She was a very interesting human being with the talent, but he was so incredibly, not unlike, "Eddie" by George C. Scott on the other hand, was very, very good. He is very hard to believe that it was he who played
Sample: She was a pretty good-looking girl. The kids liked the girls and her dad was a real good-looking family. A poor girl (played nicely by another girl)... was in a crappy marriage, not a big girl... but a decent-looking woman
Sample: She was a really good looking, even if everyone else looked bad to her. I really liked John Travolta.<br /><br />And, of course, she had a really good little little boy, who looked so different in the early years of
Sample: She was a good dancer and that role is very well done. I've seen the short film before but it's really amazing. The movie starts out to be a very emotional part of the movie. There is a great fight with th

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: I hope you will watch her because they have more than their share of the fun." (As she and her husband are not in the same room ) The film ends with one of the great comedies of the 50s, "Gimme Shelter", which had
Sample: I hope you get a lot of credit for this. Let me say this is not a movie for all films. I think its just another attempt to show the depths of cinema. The story focuses on a woman with one girl (anonymous.) (A woman like
Sample: I hope you will make a real movie with me. It is a movie I've been in about 3 days and thought a lot. I didn't believe the first one was so good, and really good, in a real way. I think I have it right
Sample: I hope you won't read "Titanic". I'm going to tell you something very important.<br /><br />Yes, we are going to see a very strange place. I'll say what we do...and that's that we're going to
Sample: I hope you never buy this film. Once its in, you will have a new experience in the cinema. You will be a movie addict to the point o

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was the last film to hit the big screen. That's an interesting statement that you just get right by going in the wrong direction. The title of the film is so misleading that its been used many times. You see, the films are trying to move us and
Sample: It was the last time I watched this film.<br /><br />I'd wager the same of you might see it today because so many other films have tried to do the same. But just because you don't know any of the new ones does not make
Sample: It was the last moment of the film in the history of "Silence from the Screen". If it were not for Richard Gere and his lovely female counterpart, the film would have survived any chance. Instead they gave him a wonderful, but completely unnecessary, sex machine
Sample: It was the last few seconds before the 'Mulan' 'WTF?' movie I remembered. And, to add to the shock factor, the movie's cover design and other "bizarre" scenes, which are used as clues or clues to the plot
Sample: It was the last nail in t

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: From the beginning, there are two different endings, however they are based on a great combination of the characters, the two antagonists and our hero, together form a great series. The best part of the series is the very funny and amazing comedy in the first, as
Sample: From the beginning) the title is nothing. The character is actually a dog or something. The whole point of the film is to drive the cat away from the point of a police dog. The way this goes, he isn't a bad guy, but not a
Sample: From the beginning to last scene the film, there is no surprise.<br /><br />The story is what it's all about. If there was any way to see it, it is the film. If there were any way to see it, it would
Sample: From the beginning he was played by Arthur Lee Smith).The film starts up with him, and the girl (Olivia Harnes) and her brother (Gus Moroder) is falling into the heart of a world that hates people.<br /><
Sample: From the beginning, this is the best film ever in my opinion - as the

Perhaps filtering comments (leaving most obscene/insulting ones) would give more visible results. Idea 1: leave couple of toxic categories, not all of them.
Idea 2: use model that was finetuned for classifying comments (toxic/non_toxic) and use this model outputs to filter out the dataset.